# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaribShahid/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring.** I chose this lane because the practical decision is how to prioritize a limited amount of human review across a large content inventory. The starter dataset has 30,000 content items, and 16,262 (54.2%) are currently marked as `down` in the starter's current-window trend label. I also found 9,961 items that are both `down` and have at least 500 impressions in the 90-day window, so there is a substantial group where movement and visibility can both matter. Over the next 7 weeks, I want to test whether observable content/search signals can produce a useful ranked review queue for actions such as refresh, expand, protect, or monitor. I will treat the starter decline label as a **proxy/current-window signal**, not as proof of future decline or proof that a refresh will improve performance.


In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

required_columns = ["content_id", "client_id", "impressions_90d", "sessions_90d", "content_age_days", "trend_direction"]
missing = [c for c in required_columns if c not in df.columns]
print(f"Starter rows: {len(df):,}")
print(f"Missing required columns: {missing}")
assert not missing
assert len(df) == 30_000


Starter rows: 30,000
Missing required columns: []


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which pages should be reviewed first for a possible refresh or another content action, based on observable evidence of visibility, movement, freshness, position, CTR, and engagement?

- **Decision:** Decide which pages deserve limited editorial/SEO review first.
- **Who acts:** An SEO/content reviewer or editor reviews the ranked queue and decides whether to refresh, expand, protect, monitor, or take another action.
- **Output:** A ranked opportunity queue with a score, suggested action, reason codes, and a confidence/evidence level.
- **Cost of a wrong call:** A false positive can waste editor time and lead to an unnecessary content change. A false negative can delay attention to a page that may need review. For high-visibility pages, a wrong recommendation can also affect a larger amount of search exposure, so the final system should support human review rather than automatic publishing changes.
- **Why data/ML can help:** A transparent rule is the first baseline. ML is only justified if several signals interact in ways that are difficult to capture with a short hand-written rule and if the learned ranking improves a decision-focused metric such as precision@K. The goal is therefore a **better prioritization decision**, not simply a trained model.


In [2]:
# Keep the decision frame visible in the executed notebook.
decision = "Which pages should be reviewed first for a possible content action?"
output = "Ranked review queue with score, action, reason codes, and confidence/evidence level"
print("Decision:", decision)
print("Output:", output)


Decision: Which pages should be reviewed first for a possible content action?
Output: Ranked review queue with score, action, reason codes, and confidence/evidence level


## 3. Quick look at the data (2-3 real numbers)

The starter CSV is a 30,000-row anonymized content inventory. These numbers make the lane worth investigating, while still leaving room to test whether the signals are genuinely useful for prioritization.

The important caveat is that the starter `down` label is calculated from the same current 30-day-vs-previous-30-day window in the starter dataset. It is useful for understanding the playground and building a baseline, but it is not yet the stronger future-looking outcome I would want for a final capstone.


In [3]:
from IPython.display import display

down_count = int((df["trend_direction"] == "down").sum())
down_rate = down_count / len(df) * 100
down_with_demand = int(((df["trend_direction"] == "down") & (df["impressions_90d"] >= 500)).sum())
low_ctr_visible = int(((df["impressions_90d"] >= 500) & df["avg_position"].between(1, 20) & (df["ctr"] < 0.5)).sum())
summary = pd.DataFrame({
    "Measure": ["Starter content items", "Current-window down items", "Down + at least 500 impressions", "Visible + low CTR review candidates"],
    "Value": [len(df), down_count, down_with_demand, low_ctr_visible],
})
display(summary)
print(f"Down share: {down_rate:.1f}%")


,Measure,Value
0,Starter content items,30000
1,Current-window down items,16262
2,Down + at least 500 impressions,9961
3,Visible + low CTR review candidates,9745


Down share: 54.2%


## 4. Careful words: what I can and can't claim

**What I can claim:** I can measure associations and ranking performance on the available anonymized data. I can report which observable signals are associated with the starter outcome, whether a baseline or model ranks the observed starter label better, and whether the resulting queue is useful as **decision support** under a stated validation design.

**What I cannot claim:** I cannot claim that a signal is a Google ranking factor, that a page is guaranteed to recover after a refresh, or that the starter model predicts future Google performance. I also cannot claim that a refresh causes recovery from this observational dataset alone. The starter `is_declining_label` is a current-window proxy derived from `trend_direction`, so `trend_direction` and `trend_pct` must not be used as features. For the capstone, I should define a future-window outcome and keep the feature window strictly before that outcome.

I will use careful language such as **observed**, **measured**, **associated with**, **directional**, and **decision-support** rather than causal or guaranteed language.


In [4]:
print("Starter label caveat: is_declining_label is not a source column here; it is derived from trend_direction == 'down'.")
print("For the capstone, the preferred design is: earlier feature window -> future outcome window, with no overlap.")
print("Decision-support claim only: a high score means 'review earlier', not 'refresh will definitely work'.")


Starter label caveat: is_declining_label is not a source column here; it is derived from trend_direction == 'down'.
For the capstone, the preferred design is: earlier feature window -> future outcome window, with no overlap.
Decision-support claim only: a high score means 'review earlier', not 'refresh will definitely work'.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.